In [ ]:
%run ./_bootstrap

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
from helpers.raw_storage import write_bytes
from helpers.ingestion_helper import ensure_raw_table, load_records_to_raw_data
from helpers.medlineplus_ingestion import (
    get_latest_medlineplus_zip_url,
    download_medlineplus_zip,
    extract_first_xml_from_zip,
    parse_medlineplus_xml_to_records,
)

In [ ]:
spark = SparkSession.builder.getOrCreate() # create a spark session
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [ ]:
ensure_raw_table(spark)

In [ ]:
ingest_date = datetime.utcnow().strftime("%Y-%m-%d")
zip_url = get_latest_medlineplus_zip_url()
print("Using MedlinePlus compressed XML from:", zip_url)

zip_bytes = download_medlineplus_zip(zip_url)

In [ ]:
raw_path = (
    f"/Volumes/workspace/med/medibot_bronze/"
    f"medlineplus/ingest_date={ingest_date}/medlineplus.zip"
)
write_bytes(raw_path, zip_bytes)
print("Saved raw MedlinePlus zip to:", raw_path)

In [ ]:
# extract xml bytes and parse to records
xml_bytes = extract_first_xml_from_zip(zip_bytes)
records = parse_medlineplus_xml_to_records(xml_bytes)
print("records:", len(records))

In [ ]:
load_records_to_raw_data(spark, records, source_value="medlineplus")

In [ ]:
dbutils.fs.ls("dbfs:/Volumes/workspace/med/medibot_bronze/medlineplus")